In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install numpy pillow torch torchvision

# IMAGE CLASSFICATION FOR CIFAR 10 DATASET

In [3]:
import numpy as np 
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [4]:
# Define transformations we want to apply to incoming data before it comes into network

# When we get image data its b/w 0 and 255 i.e., RGB values- we want to scale b/w -1 and 1, & we want them to be tensors

transform = transforms.Compose([
    transforms.ToTensor(), # Turns data to tensor and normalizes it to b/w 0 and 1 from 0 to 255
    
    # to get the vals b/w -1 and 1, we need to normalize w/ mean of 0.5 and std dev of 0.5 since we subtract 0.5 and divide by 0.5
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) # done to speed up convergence, 3 0.5s for each channel
])

In [5]:
# CIPHAR10 has 10 diff classes of imgs

# training data is downloaded to kaggle's input directory, and transform is applied
train_data = torchvision.datasets.CIFAR10(root = '/kaggle/working', train = True, transform = transform, download = True)

# testing data is downloaded to kaggle's input directory, and transform is applied
test_data = torchvision.datasets.CIFAR10(root = '/kaggle/working', train = False, transform = transform, download = True)

# Create data loaders to load the data 
train_loader = torch.utils.data.DataLoader(train_data, batch_size = 32, shuffle = True, num_workers = 2) # 2 workers used to load data in parallel in sizes of 32 and shuffled
test_loader = torch.utils.data.DataLoader(test_data, batch_size = 32, shuffle = True, num_workers = 2) # 2 workers used to load data in parallel in sizes of 32 and shuffled

100%|██████████| 170M/170M [00:02<00:00, 61.3MB/s]


### LOOK AT DATA TO SEE ITS SHAPE AND HOWS ITS STRUCTURED

In [6]:
image, label = train_data[0] # to see first image of train dataset

In [7]:
image.size() # 3 channels rgb, 32x32 px

torch.Size([3, 32, 32])

In [8]:
# Writing class labels
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Network produces a number b/w 0 and 9 representing individual classes

### NEURAL NETWORK 

In [9]:
# Defining the neural network

class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        # A kernel moves over the image and extracts features  by calculating dot product and creates feature maps. Over time kernel is trained and creates accurate feature maps
        self.conv1 = nn.Conv2d(3, 12, 5) # i/p channels is 3 defined by shape of data, 12 o/p channels (feature maps) which is result of applying 5*5 kernel or filter

        # Now we get a new shape given by ((input size - kernel size)/stride) + 1 where stride is how much our kernel moves - here default 1px. So (32-5)/1 + 1
        # New shape after convolutional layer is (12, 28, 28) or 12 channels w/ 28x28 px
        self.pool = nn.MaxPool2d(2,2) # Takes 2*2 pxs and creates 1 px out of that. Extracts most imp info leading to a division by 2 for our shape. Hence (12, 14, 14)

        self.conv2 = nn.Conv2d(12, 24, 5) # (14-5)/1 + 1. New shape is (24,10,10). We do max pooling in the feed forward nn giving (24,5,5) followed by flattening to (24*5*5)

        self.fc1 = nn.Linear(24*5*5, 120) # Dense layer or fully connected layer. I/p is 24*5*5, output is 120 neurons
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # we NEED to have 10 output neurons i.e., probability for 10 classes. Rest you can play around with

    def forward(self, x): # Applies layers onto input
        # we take i/p val and feed it thru first conv layer, result is fed to relu (0 if i/p < 0 , and x if i/p = x) to break linearity, fed to pooling layer
        x = self.pool(F.relu(self.conv1(x))) 
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # apply flatten operation
        x = F.relu(self.fc1(x)) # apply relu onto o/p of fcl (after feeding i/p thru fcl)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x     

In [10]:
net = NeuralNet() # define the network
loss_function = nn.CrossEntropyLoss() # categorical cross entropy loss function is used
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9) # optimizer is stochastic gradient descent 

In [11]:
# to train the model

for epoch in range(30):
    print(f'Training epoch {epoch}...')

    running_loss = 0.0

    for i, data in enumerate(train_loader): # get batches from train loader
        inputs, labels = data

        optimizer.zero_grad() # reset all gradients to zero

        outputs = net(inputs) # take i/ps from data feed to n/w and get o/p

        loss = loss_function(outputs, labels) # compared to desired o/ps what are our o/ps
        loss.backward() # backpropagate the loss and update gradients
        optimizer.step() # take a small step (depedning on lr) into correct direction for each param we have

        running_loss += loss.item() # add up loses for individual instances

    print(f'Loss: {running_loss / len(train_loader):.4f}') 

Training epoch 0...
Loss: 2.2694
Training epoch 1...
Loss: 1.7924
Training epoch 2...
Loss: 1.5640
Training epoch 3...
Loss: 1.4410
Training epoch 4...
Loss: 1.3361
Training epoch 5...
Loss: 1.2436
Training epoch 6...
Loss: 1.1717
Training epoch 7...
Loss: 1.1096
Training epoch 8...
Loss: 1.0594
Training epoch 9...
Loss: 1.0171
Training epoch 10...
Loss: 0.9745
Training epoch 11...
Loss: 0.9442
Training epoch 12...
Loss: 0.9051
Training epoch 13...
Loss: 0.8684
Training epoch 14...
Loss: 0.8351
Training epoch 15...
Loss: 0.8061
Training epoch 16...
Loss: 0.7770
Training epoch 17...
Loss: 0.7473
Training epoch 18...
Loss: 0.7210
Training epoch 19...
Loss: 0.6933
Training epoch 20...
Loss: 0.6701
Training epoch 21...
Loss: 0.6460
Training epoch 22...
Loss: 0.6224
Training epoch 23...
Loss: 0.5991
Training epoch 24...
Loss: 0.5766
Training epoch 25...
Loss: 0.5563
Training epoch 26...
Loss: 0.5335
Training epoch 27...
Loss: 0.5148
Training epoch 28...
Loss: 0.4896
Training epoch 29...
Los

In [12]:
# Export our model parameters
torch.save(net.state_dict(), 'trained_net.pth')

In [13]:
net = NeuralNet() # create another instance of network
net.load_state_dict(torch.load('trained_net.pth')) # load state dictionary

<All keys matched successfully>

In [14]:
# Evaluate on test data

correct = 0
total = 0

net.eval() # set network to eval mode

with torch.no_grad(): # to not do gradient computations cos we're not training model
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1) # highest activation will be our label
        total += labels.size(0)
        correct += (predicted == labels).sum().item() # counting how many have correct classfication

accuracy = 100 * correct / total

print(f'Accuracy: {accuracy}%')

Accuracy: 67.49%


In [15]:
# To test on an image from the web- to have same input format
new_transform = transforms.Compose([
    transforms.Resize((32, 32)), # to convert img to 32x32 px 
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)
    image = image.unsqueeze(0) # add extra dim - cos we want to have batch dim. Single img needs to be presented as batch for compatibility
    return image

'''
image_paths = ['example1.jpg', 'example2.jpg']
images = [load_image(img) for img in image_paths]

net.eval()

with torch.no_grad():
    for image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Prediction: {class_names[predicted.item()]}')
'''

"\nimage_paths = ['example1.jpg', 'example2.jpg']\nimages = [load_image(img) for img in image_paths]\n\nnet.eval()\n\nwith torch.no_grad():\n    for image in images:\n        output = net(image)\n        _, predicted = torch.max(output, 1)\n        print(f'Prediction: {class_names[predicted.item()]}')\n"